In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [34]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print(train.shape)
print(test.shape)

test.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [35]:
import pandas as pd

# check files
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(root, file)

/kaggle/input/competitions/smart-mcq-solver-challenge sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge test.csv


In [36]:
submission = pd.DataFrame({
    "id": test["id"],
    "answer": "A"
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,answer
0,1,A
1,2,A
2,3,A
3,4,A
4,5,A


In [37]:
import os
print(os.listdir())

['submission.csv', '.virtual_documents']


In [38]:
!pip show torch

Name: torch
Version: 2.13.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, easyocr, fastai, kornia, peft, pytorch-ignite, pytorch-lightning, sentence-transformers, timm, torchdata, torchmetrics, torchvision


In [39]:
!pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [40]:
!pip install torch torchvision torchaudio

  Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl (526.6 MB)
Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.7 MB)
Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.8 MB)


In [41]:
import torch
import pandas as pd
import transformers

print(torch.cuda.is_available())

True


In [42]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [43]:
import re

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]',' ',text)
    text = re.sub(r'\s+',' ',text)
    return text

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

In [45]:
from gensim.models import Word2Vec

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
cols = ["prompt", "A", "B", "C", "D", "E"]

for col in cols:
    train[col] = train[col].fillna("").apply(clean)

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
1,2,what is accelerator based light ion fusion,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,A
2,3,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing,C
3,4,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
4,5,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,A


In [48]:
train["text"] = (
    train["prompt"] +
    " A: " + train["A"] +
    " B: " + train["B"] +
    " C: " + train["C"] +
    " D: " + train["D"] +
    " E: " + train["E"]
)

train["text"].head()

0    pick the best possible answer what is martin h...
1    what is accelerator based light ion fusion  A:...
2    determine the correct option what is the term ...
3    select the most accurate option what is martin...
4    identify the correct statement what is the con...
Name: text, dtype: object

In [49]:
X = tfidf.fit_transform(train["text"])

print(X.shape)

(2000, 2939)


In [50]:
sentences = [text.split() for text in train["text"]]

In [51]:
w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
print(w2v.wv["pick"][:10])

[ 0.30095133  0.49234685 -0.23658036  0.29804134 -0.44537583 -0.64816415
  0.7718764   1.3314872  -0.21982683 -0.4446926 ]


In [52]:
# %% [markdown]
# **Model 1 (From Scratch): Word2Vec + Feedforward Classifier**

# %%
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# ---- Build sentence-level features from your own Word2Vec embeddings ----
def avg_w2v(text, model, dim=100):
    tokens = text.split()
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

def row_to_features(row, w2v_model, dim=100):
    """Concatenate: prompt embedding + each option embedding (A-E)."""
    prompt_vec = avg_w2v(row["prompt"], w2v_model, dim)
    option_vecs = [avg_w2v(row[c], w2v_model, dim) for c in ["A","B","C","D","E"]]
    return np.concatenate([prompt_vec] + option_vecs)  # shape: (dim*6,)

feat_dim = 100
X_scratch = np.stack([row_to_features(r, w2v, feat_dim) for _, r in train.iterrows()])
y_scratch = train["answer"].map({"A":0,"B":1,"C":2,"D":3,"E":4}).values

X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_scratch, y_scratch, test_size=0.1, random_state=42, stratify=y_scratch
)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.long)
X_val_t   = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t   = torch.tensor(y_val_s, dtype=torch.long)

# ---- The model itself: plain nn.Module, random init, no pretrained weights ----
class ScratchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
scratch_model = ScratchMLP(input_dim=feat_dim * 6).to(device)
optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3)

# %%
# ---- W&B run for THIS model (separate from the others) ----
import wandb

wandb_run_scratch = wandb.init(
    project="YourRollNo-t22026",
    name="scratch-mlp-word2vec",
    reinit=True
)

# %%
# ---- Train loop with metric logging ----
from sklearn.metrics import accuracy_score, f1_score

epochs = 15
batch_size = 32
n_train = X_train_t.shape[0]

for epoch in range(epochs):
    scratch_model.train()
    perm = torch.randperm(n_train)
    total_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)

        optimizer.zero_grad()
        logits = scratch_model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    train_loss = total_loss / n_train

    # ---- eval ----
    scratch_model.eval()
    with torch.no_grad():
        val_logits = scratch_model(X_val_t.to(device))
        val_preds = torch.argmax(val_logits, dim=1).cpu().numpy()
        val_acc = accuracy_score(y_val_s, val_preds)
        val_f1 = f1_score(y_val_s, val_preds, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "eval/accuracy": val_acc,
        "eval/f1": val_f1,
    })

    print(f"Epoch {epoch}: loss={train_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

wandb_run_scratch.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2000043 (24f2000043-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 0: loss=1.4770 val_acc=0.4500 val_f1=0.4504
Epoch 1: loss=1.0735 val_acc=0.7250 val_f1=0.7225
Epoch 2: loss=0.6357 val_acc=0.9350 val_f1=0.9351
Epoch 3: loss=0.3205 val_acc=1.0000 val_f1=1.0000
Epoch 4: loss=0.1686 val_acc=1.0000 val_f1=1.0000
Epoch 5: loss=0.0902 val_acc=1.0000 val_f1=1.0000
Epoch 6: loss=0.0651 val_acc=1.0000 val_f1=1.0000
Epoch 7: loss=0.0370 val_acc=1.0000 val_f1=1.0000
Epoch 8: loss=0.0351 val_acc=1.0000 val_f1=1.0000
Epoch 9: loss=0.0266 val_acc=1.0000 val_f1=1.0000
Epoch 10: loss=0.0195 val_acc=1.0000 val_f1=1.0000
Epoch 11: loss=0.0254 val_acc=1.0000 val_f1=1.0000
Epoch 12: loss=0.0165 val_acc=1.0000 val_f1=1.0000
Epoch 13: loss=0.0094 val_acc=1.0000 val_f1=1.0000
Epoch 14: loss=0.0131 val_acc=1.0000 val_f1=1.0000


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
eval/accuracy,▁▅▇████████████
eval/f1,▁▄▇████████████
train_loss,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
epoch,14
eval/accuracy,1
eval/f1,1
train_loss,0.01312


In [53]:
prompt = train.loc[0, "prompt"]

options = [
    train.loc[0, "A"],
    train.loc[0, "B"],
    train.loc[0, "C"],
    train.loc[0, "D"],
    train.loc[0, "E"]
]

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform([prompt] + options)

similarity = cosine_similarity(vectors[0:1], vectors[1:])

print(similarity)

[[0.19148327 0.21487352 0.39850777 0.36043491 0.11200832]]


**MILESTONE 1**

In [54]:
from transformers import AutoTokenizer, AutoModel
import torch

In [55]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [56]:
text = f"""
Question: {train.loc[0,'prompt']}

A. {train.loc[0,'A']}
B. {train.loc[0,'B']}
C. {train.loc[0,'C']}
D. {train.loc[0,'D']}
E. {train.loc[0,'E']}
"""

print(text)


Question: pick the best possible answer what is martin heidegger s view on the relationship between time and human existence among the listed options 

A. martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond one s own time 
B. martin heidegger believes that humans do not exist inside time but that they are time the relationship to the past is a present awareness of having been and the relationship to the future involves anticipating a potential possibility task or engagement 
C. martin heidegger does not believe in the existence of time or that it has any effect on human consciousness the relationship to the past and the future is insignificant and human existence is solely based on the present 
D. martin heidegger believes that the relationship betwee

In [57]:
encoding = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

print(encoding)

{'input_ids': tensor([[  101,  3160,  1024,  4060,  1996,  2190,  2825,  3437,  2054,  2003,
          3235,  2002,  5178, 13327,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  2426,  1996,  3205,  7047,  1037,  1012,
          3235,  2002,  5178, 13327,  7164,  2008,  4286,  4839,  2306,  1037,
          2051, 22961,  2008,  2003, 10709,  1998,  2515,  2025,  2031,  1037,
          4225,  2927,  2030,  2203,  1996,  3276,  2000,  1996,  2627,  7336,
         21894,  2009,  2004,  1037,  3439,  3690,  1998,  1996,  3276,  2000,
          1996,  2925,  7336,  4526,  1037,  2088,  2008,  2097, 18094,  3458,
          2028,  1055,  2219,  2051,  1038,  1012,  3235,  2002,  5178, 13327,
          7164,  2008,  4286,  2079,  2025,  4839,  2503,  2051,  2021,  2008,
          2027,  2024,  2051,  1996,  3276,  2000,  1996,  2627,  2003,  1037,
          2556,  7073,  1997,  2383,  2042,  1998,  1996,  3276,  2000,  1996,
          2925,  7336, 26481,  1037,  

In [58]:
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
with torch.no_grad():
    outputs = model(**encoding)

print(outputs.last_hidden_state.shape)

torch.Size([1, 512, 768])


In [60]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = ["A", "B", "C", "D", "E"]

result = classifier(
    train.loc[0, "prompt"],
    candidate_labels
)

print(result)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'pick the best possible answer what is martin heidegger s view on the relationship between time and human existence among the listed options ', 'labels': ['A', 'B', 'D', 'C', 'E'], 'scores': [0.30863043665885925, 0.20200763642787933, 0.18599382042884827, 0.15286773443222046, 0.15050038695335388]}


In [61]:
candidate_labels = [
    train.loc[0, "A"],
    train.loc[0, "B"],
    train.loc[0, "C"],
    train.loc[0, "D"],
    train.loc[0, "E"]
]

result = classifier(
    train.loc[0, "prompt"],
    candidate_labels
)

print(result)

{'sequence': 'pick the best possible answer what is martin heidegger s view on the relationship between time and human existence among the listed options ', 'labels': ['martin heidegger believes that humans do not exist inside time but that they are time the relationship to the past is a present awareness of having been and the relationship to the future involves anticipating a potential possibility task or engagement ', 'martin heidegger believes that the relationship between time and human existence is cyclical the past and present are interconnected and the future is predetermined human beings do not have free will ', 'martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond one s own time ', 'martin heidegger believes that time is an illusion and the pa

In [62]:
!pip install -q faiss-cpu sentence-transformers

In [63]:
import torch
print(torch.__version__)

2.13.0+cu130


In [64]:
!pip install faiss-cpu

In [66]:
!pip install -q "sentence-transformers<3.3" --force-reinstall --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 7.9 MB/s eta 0:00:00


In [67]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [68]:
print("Creating Knowledge Base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Knowledge Base Size:", len(kb))

Creating Knowledge Base...
Knowledge Base Size: 2000


In [69]:
print("Loading Embedding Model...")

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [70]:
print("Generating Embeddings...")

kb_embeddings = embedder.encode(
    kb,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(kb_embeddings.shape)

Generating Embeddings...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


In [71]:
dimension = kb_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(kb_embeddings)

print("Vectors in Index:", index.ntotal)

Vectors in Index: 2000


In [72]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [73]:
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

**MILESTONE 3**

In [74]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base successfully created


In [75]:

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [76]:
# Q1

result = zs(
    prompt_150,
    candidate_labels=labels_150
)

correct_score = result["scores"][result["labels"].index(ans_150)]

print("Ground Truth Answer:", ans_150)
print("Predicted Probability:", round(correct_score, 3))

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Predicted Probability: 0.384


In [77]:
# Q2

query_embedding = model.encode(
    [prompt_150],
    convert_to_numpy=True,
    show_progress_bar=False
)

distances, retrieved_indices = index.search(query_embedding, k=10)

print("Retrieved Indices:", retrieved_indices[0])

if 150 in retrieved_indices[0]:
    rank = np.where(retrieved_indices[0] == 150)[0][0] + 1
    print("Rank of true document:", rank)
else:
    print("True document not found in Top-10")

Retrieved Indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of true document: 10


In [78]:
# Q3

from sentence_transformers import CrossEncoder
import numpy as np

# Load Cross Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Get the top-10 documents retrieved by FAISS in Q2
docs_10 = [kb[i] for i in retrieved_indices[0]]

# Create (question, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Get Cross-Encoder scores
ce_scores = cross_encoder.predict(pairs)

# Sort documents by Cross-Encoder score (highest first)
sorted_order = np.argsort(ce_scores)[::-1]

# Reordered document indices
reranked_indices = retrieved_indices[0][sorted_order]

print("Original FAISS Indices :", retrieved_indices[0])
print("Cross-Encoder Scores   :", ce_scores)
print("Reranked Indices       :", reranked_indices)

# Find the rank of the true document (index 150)
if 150 in reranked_indices:
    rank = np.where(reranked_indices == 150)[0][0] + 1
    print("\nRank of true document after reranking:", rank)
else:
    print("\nTrue document not found in Top-10 after reranking.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original FAISS Indices : [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Cross-Encoder Scores   : [4.6602316 4.6602316 4.737523  4.737523  4.687043  4.752577  4.752577
 4.752577  4.7072477 4.758512 ]
Reranked Indices       : [ 150 1906  847 1693 1269 1532  168  576 1701  663]

Rank of true document after reranking: 1


In [79]:
# Q4

# Get row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode(
    [prompt_42],
    convert_to_numpy=True,
    show_progress_bar=False
)

# Retrieve Top-5 documents from FAISS
distances, retrieved_indices = index.search(query_embedding, k=5)

# Concatenate the retrieved documents
context = " ".join([kb[i] for i in retrieved_indices[0]])

# Create the RAG string
rag_text = f"Context: {context} Question: {prompt_42}"

# Tokenize using the BERT tokenizer WITHOUT truncation
tokenized = tokenizer(
    rag_text,
    truncation=False,
    add_special_tokens=True
)

# Count total tokens
total_tokens = len(tokenized["input_ids"])

print("Retrieved Indices:", retrieved_indices[0])
print("Total Tokens:", total_tokens)

Retrieved Indices: [1581 1760   42  241  439]
Total Tokens: 216


In [80]:
# Q5

# Retrieve the true document for row 150
true_document = kb[150]

# Create the RAG input
rag_text = f"Context: {true_document} Question: {prompt_150}"

# Run zero-shot classification
result = zs(
    rag_text,
    candidate_labels=labels_150
)

# Extract the probability of the ground-truth correct option
correct_score = result["scores"][result["labels"].index(ans_150)]

print("True Document:", true_document)
print("Ground Truth Answer:", ans_150)
print("Probability of Correct Option:", round(correct_score, 3))

True Document: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability of Correct Option: 0.989


In [81]:
# Q6

# Use an unrelated document (KB index 999) as adversarial context
wrong_document = kb[999]

# Create the Adversarial RAG string
adversarial_rag = f"Context: {wrong_document} Question: {prompt_150}"

# Run zero-shot classification
result = zs(
    adversarial_rag,
    candidate_labels=labels_150
)

# Extract the probability of the ground-truth correct option
correct_score = result["scores"][result["labels"].index(ans_150)]

print("Wrong Context Document:", wrong_document)
print("Ground Truth Answer:", ans_150)
print("Probability of Correct Option:", round(correct_score, 3))

Wrong Context Document: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.
Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability of Correct Option: 0.529


In [82]:
# Q7

hits = 0

for i in range(100):
    # Get prompt and correct answer
    row = train.iloc[i]
    prompt = str(row["prompt"])
    correct_doc = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # Retrieve Top-5 documents
    _, retrieved_indices = index.search(query_embedding, k=5)

    # Retrieved documents
    retrieved_docs = [kb[idx] for idx in retrieved_indices[0]]

    # Check if the exact correct document is among the retrieved documents
    if correct_doc in retrieved_docs:
        hits += 1

# Calculate Hit Rate
hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate (%):", round(hit_rate, 1))

Hits: 73
Hit Rate (%): 73.0


In [83]:
# Q8

import numpy as np

def map_at_3(actual, predicted):
    """
    actual: correct option letter (e.g., 'A')
    predicted: list of top-3 predicted option letters
    """
    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)
    return 0.0

scores = []

for i in range(20):

    # -----------------------
    # Get current row
    # -----------------------
    row = train.iloc[i]

    prompt = str(row["prompt"])

    option_texts = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    option_letters = ["A", "B", "C", "D", "E"]

    correct_letter = row["answer"]

    # -----------------------
    # Retrieve Top-5
    # -----------------------
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    _, retrieved_indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in retrieved_indices[0]]

    # -----------------------
    # Cross-Encoder Reranking
    # -----------------------
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # -----------------------
    # Build RAG Prompt
    # -----------------------
    rag_text = f"Context: {best_doc} Question: {prompt}"

    # -----------------------
    # Zero-shot Prediction
    # -----------------------
    result = zs(
        rag_text,
        candidate_labels=option_texts
    )

    # -----------------------
    # Convert predicted option text back to option letter
    # -----------------------
    ranked_letters = []

    for predicted_option in result["labels"]:
        idx = option_texts.index(predicted_option)
        ranked_letters.append(option_letters[idx])

    top3 = ranked_letters[:3]

    # -----------------------
    # MAP@3
    # -----------------------
    scores.append(map_at_3(correct_letter, top3))

# Final MAP@3
final_map3 = np.mean(scores)

print("Average MAP@3:", round(final_map3, 3))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Average MAP@3: 0.975


In [84]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["text"] = train.apply(
    lambda row:
    f"""
    Question: {row['prompt']}
    Option A: {row['A']}
    Option B: {row['B']}
    Option C: {row['C']}
    Option D: {row['D']}
    Option E: {row['E']}
    """,
    axis=1
)

train["label"] = train["answer"].map(label_map)

In [85]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01


In [86]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [87]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base"
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [88]:
def tokenize_function(example):
    
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [89]:
dataset = Dataset.from_pandas(
    train[["text", "label"]]
)

In [90]:
dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [91]:
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [92]:
# Rebuild from the pandas dataframe + retokenize (safe, avoids any leftover state issues)
dataset = Dataset.from_pandas(train[["text", "label"]])
dataset = dataset.map(tokenize_function, batched=True)

# Now split — dataset here is guaranteed to be a plain Dataset
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_dataset["train"]
val_ds = split_dataset["test"]

print(train_ds)
print(val_ds)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1800
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})


In [93]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=5
)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

In [94]:
!pip install -q -U torchao

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.6 MB/s eta 0:00:0000:0100:01


In [95]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"]  # DeBERTa-v3 attention proj layers
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory
W0719 18:59:39.130000 58 torch/utils/_pytree.py:630] <enum 'KernelPreferen

trainable params: 298,757 || all params: 184,724,746 || trainable%: 0.1617


In [96]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}

In [97]:
model.print_trainable_parameters()

trainable params: 298,757 || all params: 184,724,746 || trainable%: 0.1617


In [98]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,   # drop to 4 if you hit CUDA OOM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=20,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)



In [99]:
import wandb

wandb.init(
    project="YourRollNo-t22026",   # replace with your actual W&B project name
    name="milestone4-deberta-lora"
)

In [100]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=False,              # <-- disabled, was causing the dtype mismatch
    logging_steps=20,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [101]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,7.149877,3.138468,0.315000,0.261580
2,6.933684,3.035775,0.260000,0.169994
3,6.481470,3.027732,0.315000,0.239900


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval/accuracy,█▁█
eval/f1,█▁▆
eval/loss,█▂▁
eval/runtime,▁█▇
eval/samples_per_second,█▁▂
eval/steps_per_second,█▁▂
train/epoch,▁▂▃▃▄▅▅▆▇███
train/global_step,▁▂▃▃▄▅▅▆▇▇██
train/grad_norm,▁█▅▆▅▅▂▁
train/learning_rate,█▇▆▅▄▃▂▁
+1,...


Q1 (milestone4)

In [102]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train["label"] = train["answer"].map(label_map)

print("Q1 Answer:", train.loc[150, "label"])

Q1 Answer: 2


In [103]:
prompt_0 = str(train.loc[0, "prompt"])
option_B_0 = str(train.loc[0, "B"])

formatted_input = prompt_0 + " [SEP] " + option_B_0

print("Q2 Answer:", len(formatted_input))

Q2 Answer: 407


In [104]:
from transformers import AutoTokenizer

mc_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row0 = train.loc[0]
prompt_0 = str(row0["prompt"])
options_0 = [str(row0["A"]), str(row0["B"]), str(row0["C"]), str(row0["D"]), str(row0["E"])]

# Each choice = prompt + [SEP] + option
first_sentences = [prompt_0] * 5
second_sentences = options_0

encoding = mc_tokenizer(
    first_sentences,
    second_sentences,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [1, 5, 128] — 1 question, 5 choices, 128 tokens
input_ids = encoding["input_ids"].view(1, 5, 128)

print("Shape:", input_ids.shape)
print("Q3 Answer:", input_ids.shape[1])

Shape: torch.Size([1, 5, 128])
Q3 Answer: 5


In [105]:
batch_rows = train.iloc[:16]

first_sentences = []
second_sentences = []

for _, row in batch_rows.iterrows():
    prompt = str(row["prompt"])
    options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
    first_sentences.extend([prompt] * 5)
    second_sentences.extend(options)

batch_encoding = mc_tokenizer(
    first_sentences,
    second_sentences,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

batch_input_ids = batch_encoding["input_ids"].view(16, 5, 128)

print("Shape:", batch_input_ids.shape)
print("Q4 Answer:", batch_input_ids.numel())  # total token positions = 16*5*128

Shape: torch.Size([16, 5, 128])
Q4 Answer: 10240


In [106]:
from transformers import AutoModelForMultipleChoice
import torch

mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Reshape encoding for row 0 back to [1, 5, 128] for model input
row0_encoding = mc_tokenizer(
    [prompt_0]*5,
    options_0,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

model_input = {k: v.unsqueeze(0) for k, v in row0_encoding.items()}  # add batch dim -> [1,5,128]

with torch.no_grad():
    outputs = mc_model(**model_input)

print("Logits shape:", outputs.logits.shape)
print("Q5 Answer:", outputs.logits.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Q5 Answer: 5


In [107]:
label_0 = torch.tensor([train.loc[0, "label"]])

outputs_with_loss = mc_model(**model_input, labels=label_0)

print("Loss:", outputs_with_loss.loss)
print("Q6 Answer:", outputs_with_loss.loss.dim())  # scalar -> 0 dimensions

Loss: tensor(1.6104, grad_fn=<NllLossBackward0>)
Q6 Answer: 0


In [108]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config_mc = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

mc_model_lora = get_peft_model(mc_model, lora_config_mc)

trainable_params = sum(p.numel() for p in mc_model_lora.parameters() if p.requires_grad)
print("Q7 Answer:", trainable_params)

Q7 Answer: 295681


In [109]:
from datasets import Dataset

def preprocess_mc(examples):
    first_sentences = []
    second_sentences = []
    labels = []

    for i in range(len(examples["prompt"])):
        prompt = str(examples["prompt"][i])
        options = [str(examples["A"][i]), str(examples["B"][i]),
                   str(examples["C"][i]), str(examples["D"][i]), str(examples["E"][i])]
        first_sentences.extend([prompt] * 5)
        second_sentences.extend(options)
        labels.append(examples["label"][i])

    tokenized = mc_tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Reshape flat lists into [num_rows, 5, seq_len]
    input_ids = [tokenized["input_ids"][i*5:(i+1)*5] for i in range(len(examples["prompt"]))]
    attention_mask = [tokenized["attention_mask"][i*5:(i+1)*5] for i in range(len(examples["prompt"]))]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

subset_100 = train.iloc[:100].reset_index(drop=True)
hf_dataset_100 = Dataset.from_pandas(subset_100[["prompt", "A", "B", "C", "D", "E", "label"]])

mc_dataset = hf_dataset_100.map(preprocess_mc, batched=True, remove_columns=hf_dataset_100.column_names)

print("input_ids shape for item 0:", np.array(mc_dataset[0]["input_ids"]).shape)
print("Q8 Answer:", np.array(mc_dataset[0]["input_ids"]).shape[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape for item 0: (5, 128)
Q8 Answer: 5


In [111]:
subset_32 = train.iloc[:32].reset_index(drop=True)
hf_dataset_32 = Dataset.from_pandas(subset_32[["prompt", "A", "B", "C", "D", "E", "label"]])

def preprocess_mc_64(examples):
    first_sentences = []
    second_sentences = []
    labels = []
    for i in range(len(examples["prompt"])):
        prompt = str(examples["prompt"][i])
        options = [str(examples["A"][i]), str(examples["B"][i]),
                   str(examples["C"][i]), str(examples["D"][i]), str(examples["E"][i])]
        first_sentences.extend([prompt] * 5)
        second_sentences.extend(options)
        labels.append(examples["label"][i])
    tokenized = mc_tokenizer(
        first_sentences, second_sentences,
        padding="max_length", truncation=True, max_length=64
    )
    input_ids = [tokenized["input_ids"][i*5:(i+1)*5] for i in range(len(examples["prompt"]))]
    attention_mask = [tokenized["attention_mask"][i*5:(i+1)*5] for i in range(len(examples["prompt"]))]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

mc_dataset_32 = hf_dataset_32.map(preprocess_mc_64, batched=True, remove_columns=hf_dataset_32.column_names)
mc_dataset_32.set_format("torch")

# Fresh model + LoRA for this tiny run
base_mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
tiny_lora_model = get_peft_model(base_mc_model, lora_config_mc)

import wandb
wandb.init(project="YourRollNo-t22026", name="bert-mc-lora-run", reinit=True)

tiny_args = TrainingArguments(
    output_dir="./tiny_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    eval_strategy="steps",   # was "no" — now it will actually evaluate
    eval_steps=2,
    report_to="wandb",
    run_name="bert-mc-lora-run",
)

tiny_trainer = Trainer(
    model=tiny_lora_model,
    args=tiny_args,
    train_dataset=mc_dataset_32,
    eval_dataset=mc_dataset_32,        # or a held-out split
    compute_metrics=compute_metrics,   # reuse the fn you already defined
)

train_result = tiny_trainer.train()   # only once now
wandb.finish()                        # moved to AFTER train()

print("Q9 Answer (global_step):", tiny_trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Accuracy,F1
2,3.296607,3.198220,0.156250,0.141931
4,3.123330,3.198159,0.156250,0.141931


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval/accuracy,▁▁
eval/f1,▁▁
eval/loss,█▁
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
train/epoch,▁▃▃▆███
train/global_step,▁▃▃▆███
train/grad_norm,▇▁█▁
train/learning_rate,█▆▃▁
+1,...


Q9 Answer (global_step): 4


In [ ]:
device = next(tiny_lora_model.parameters()).device
model_input_device = {k: v.to(device) for k, v in model_input.items()}

with torch.no_grad():
    outputs = tiny_lora_model(**model_input_device)
    probs = F.softmax(outputs.logits, dim=1)

prob_E = probs[0][4].item()
print("Q10 Answer:", round(prob_E, 4))

**MILESTONE 5**

In [112]:
import torch
import torch.nn.functional as F
import numpy as np

def get_mc_probs(model, tokenizer, df, max_length=128, device="cuda"):
    """Returns an array of shape [num_rows, 5] with softmax probabilities for A-E"""
    model.eval()
    model.to(device)
    all_probs = []

    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]

        encoding = tokenizer(
            [prompt] * 5,
            options,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        model_input = {k: v.unsqueeze(0).to(device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**model_input)
            probs = F.softmax(outputs.logits, dim=1).cpu().numpy()[0]

        all_probs.append(probs)

    return np.array(all_probs)

In [113]:
mc_test_probs = get_mc_probs(tiny_lora_model, mc_tokenizer, test)
print(mc_test_probs.shape)   # should be [num_test_rows, 5]

(500, 5)


In [114]:
def get_zeroshot_probs(df):
    all_probs = []
    label_letters = ["A", "B", "C", "D", "E"]

    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]

        result = zs(prompt, candidate_labels=options)

        # result gives labels sorted by score — realign back to A-E order
        probs = np.zeros(5)
        for label, score in zip(result["labels"], result["scores"]):
            idx = options.index(label)
            probs[idx] = score

        all_probs.append(probs)

    return np.array(all_probs)

zeroshot_test_probs = get_zeroshot_probs(test)

In [115]:
def get_tfidf_probs(df):
    all_probs = []

    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]

        vectorizer = TfidfVectorizer()
        vectors = vectorizer.fit_transform([prompt] + options)
        sims = cosine_similarity(vectors[0:1], vectors[1:])[0]

        # normalize similarities into pseudo-probabilities
        probs = sims / (sims.sum() + 1e-9)
        all_probs.append(probs)

    return np.array(all_probs)

tfidf_test_probs = get_tfidf_probs(test)

In [116]:
# Adjust weights based on each model's individual validation performance
weights = {
    "mc_lora": 0.6,
    "zeroshot": 0.25,
    "tfidf": 0.15
}

ensemble_probs = (
    weights["mc_lora"] * mc_test_probs +
    weights["zeroshot"] * zeroshot_test_probs +
    weights["tfidf"] * tfidf_test_probs
)

print(ensemble_probs.shape)

(500, 5)


In [117]:
label_letters = ["A", "B", "C", "D", "E"]

final_predictions = []

for probs in ensemble_probs:
    top3_idx = np.argsort(probs)[::-1][:3]
    top3_letters = " ".join([label_letters[i] for i in top3_idx])
    final_predictions.append(top3_letters)

submission = pd.DataFrame({
    "id": test["id"],
    "answer": final_predictions
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,answer
0,1,A D C
1,2,B A D
2,3,D A B
3,4,E C D
4,5,D C B


In [118]:
def map_at_3(actual_letter, predicted_letters):
    if actual_letter in predicted_letters:
        return 1.0 / (predicted_letters.index(actual_letter) + 1)
    return 0.0

# Example: evaluate on a validation subset where you have ground truth
val_subset = train.iloc[1800:2000]  # or however you split earlier

mc_val_probs = get_mc_probs(tiny_lora_model, mc_tokenizer, val_subset)
# ... get other models' val probs similarly ...

ensemble_val_probs = weights["mc_lora"] * mc_val_probs  # + other terms

scores = []
for i, (_, row) in enumerate(val_subset.iterrows()):
    top3_idx = np.argsort(ensemble_val_probs[i])[::-1][:3]
    top3 = [label_letters[j] for j in top3_idx]
    scores.append(map_at_3(row["answer"], top3))

print("Ensemble Validation MAP@3:", np.mean(scores))

Ensemble Validation MAP@3: 0.3241666666666667


In [119]:
trainer.save_model("./deberta_finetuned")
tokenizer.save_pretrained("./deberta_finetuned")

deberta_path = "./deberta_finetuned"

In [120]:
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=5)
deberta_model = PeftModel.from_pretrained(base_model, "./deberta_finetuned").to(device)
deberta_model.eval()

deberta_tokenizer = AutoTokenizer.from_pretrained("./deberta_finetuned")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

In [121]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

roberta_tokenizer_ft = AutoTokenizer.from_pretrained("roberta-base")
roberta_model_ft = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

def tokenize_roberta(example):
    return roberta_tokenizer_ft(example["text"], truncation=True, padding="max_length", max_length=512)

roberta_dataset = Dataset.from_pandas(train[["text", "label"]])
roberta_dataset = roberta_dataset.map(tokenize_roberta, batched=True)
roberta_split = roberta_dataset.train_test_split(test_size=0.1, seed=42)

wandb.init(project="YourRollNo-t22026", name="roberta-base-run", reinit=True)

roberta_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    fp16=False,
    logging_steps=20,
    report_to="wandb",
    run_name="roberta-base-run",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)
roberta_trainer = Trainer(
    model=roberta_model_ft,
    args=roberta_args,
    train_dataset=roberta_split["train"],
    eval_dataset=roberta_split["test"],
    compute_metrics=compute_metrics,
)

roberta_trainer.train()
wandb.finish()

roberta_trainer.save_model("./roberta_finetuned")
roberta_tokenizer_ft.save_pretrained("./roberta_finetuned")

roberta_path = "./roberta_finetuned"

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:625: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval/accuracy,▁██
eval/f1,▁██
eval/loss,█▁▁
eval/runtime,█▁▁
eval/samples_per_second,▁██
eval/steps_per_second,▁██
train/epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇███
train/global_step,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇███
train/grad_norm,▂▂▂▄▄▆▄▄█▂▄▁▁▁▁▁
train/learning_rate,██▇▇▆▆▅▅▄▄▃▃▂▂▁▁
+1,...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [122]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# Replace these paths with wherever you saved your fine-tuned checkpoints
# (e.g. "./results/checkpoint-XXX" or a Kaggle Hub model path you uploaded)
deberta_path = "./deberta_finetuned"   # loaded via PeftModel as shown above
roberta_path = "./roberta_finetuned"   # fine-tuned roberta-base

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_path)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_path, num_labels=5).to(device)
deberta_model.eval()

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_path)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_path, num_labels=5).to(device)
roberta_model.eval()

label_letters = ["A", "B", "C", "D", "E"]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

Loading weights: 0it [00:00, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: ./deberta_finetuned
Key                                                                                             | Status     | 
------------------------------------------------------------------------------------------------+------------+-
base_model.model.deberta.encoder.layer.{0...11}.attention.self.query_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...11}.attention.self.query_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...11}.attention.self.value_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...11}.attention.self.value_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.classifier.bias                                                                | UNEXPECTED | 
base_model.model.classifier.weight                                                              | UNEXPECTED | 
deberta.encoder.layer.{0...11}.

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [123]:
def build_text(row, prefix=""):
    return f"""{prefix}
    Question: {row['prompt']}
    Option A: {row['A']}
    Option B: {row['B']}
    Option C: {row['C']}
    Option D: {row['D']}
    Option E: {row['E']}
    """

def get_probs(model, tokenizer, text, max_length=512):
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=max_length, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    return probs

In [124]:
row25 = test.iloc[25] if "test" in dir() else train.iloc[25]  # use test.csv per instructions

text_25 = build_text(row25)

deberta_probs_25 = get_probs(deberta_model, deberta_tokenizer, text_25)

top_idx = np.argmax(deberta_probs_25)
print(f"Q1 Answer: {label_letters[top_idx]}, probability of {label_letters[top_idx]} = {round(deberta_probs_25[top_idx], 4)}")

Q1 Answer: B, probability of B = 0.25146484375


In [125]:
roberta_probs_25 = get_probs(roberta_model, roberta_tokenizer, text_25)

avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2

top_idx_avg = np.argmax(avg_probs_25)
print(f"Q2 Answer: {label_letters[top_idx_avg]}")

Q2 Answer: E


In [126]:
weighted_probs_25 = 0.7 * deberta_probs_25 + 0.3 * roberta_probs_25

top_idx_weighted = np.argmax(weighted_probs_25)
print(f"Q3 Answer: {label_letters[top_idx_weighted]}")

Q3 Answer: E


In [127]:
top3_idx_25 = np.argsort(weighted_probs_25)[::-1][:3]
top3_string_25 = " ".join([label_letters[i] for i in top3_idx_25])
print(f"Q4 Answer: {top3_string_25}")

Q4 Answer: E B A


In [128]:
predictions = []

for _, row in test.iterrows():
    text = build_text(row)
    d_probs = get_probs(deberta_model, deberta_tokenizer, text)
    r_probs = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs = 0.7 * d_probs + 0.3 * r_probs

    top3_idx = np.argsort(w_probs)[::-1][:3]
    top3 = " ".join([label_letters[i] for i in top3_idx])
    predictions.append(top3)

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions   # note: column name is "prediction" per this question's spec
})

submission.to_csv("submission.csv", index=False)

print("Q5 Answer:", len(submission))

Q5 Answer: 500


In [129]:
different_count = 0

for i in range(50):
    row = test.iloc[i]

    original_text = build_text(row)
    augmented_text = build_text(row, prefix="Answer the following multiple-choice question carefully:")

    probs_original = get_probs(deberta_model, deberta_tokenizer, original_text)
    probs_augmented = get_probs(deberta_model, deberta_tokenizer, augmented_text)

    avg_tta_probs = (probs_original + probs_augmented) / 2

    top1_original = np.argmax(probs_original)
    top1_tta = np.argmax(avg_tta_probs)

    if top1_original != top1_tta:
        different_count += 1

print("Q6 Answer:", different_count)

Q6 Answer: 8


In [130]:
diff_count_top1 = 0

deberta_probs_list = []
weighted_probs_list = []

for i in range(100):
    row = test.iloc[i]
    text = build_text(row)

    d_probs = get_probs(deberta_model, deberta_tokenizer, text)
    r_probs = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs = 0.7 * d_probs + 0.3 * r_probs

    deberta_probs_list.append(d_probs)
    weighted_probs_list.append(w_probs)

    if np.argmax(d_probs) != np.argmax(w_probs):
        diff_count_top1 += 1

print("Q7 Answer:", diff_count_top1)

Q7 Answer: 79


In [131]:
positive_gain_count = 0

for i in range(100):
    d_conf = np.max(deberta_probs_list[i])
    w_conf = np.max(weighted_probs_list[i])

    gain = w_conf - d_conf
    if gain > 0:
        positive_gain_count += 1

print("Q8 Answer:", positive_gain_count)

Q8 Answer: 100


In [132]:
top3_change_count = 0

for i in range(100):
    d_top3 = list(np.argsort(deberta_probs_list[i])[::-1][:3])
    w_top3 = list(np.argsort(weighted_probs_list[i])[::-1][:3])

    if d_top3 != w_top3:   # order matters, per the A C D vs A D C example
        top3_change_count += 1

print("Q9 Answer:", top3_change_count)

Q9 Answer: 79


In [133]:
def map_at_3(actual_letter, predicted_letters):
    if actual_letter in predicted_letters:
        return 1.0 / (predicted_letters.index(actual_letter) + 1)
    return 0.0

scores = []

for i in range(100):
    row = train.iloc[i]   # use train.csv since it has ground-truth 'answer' column
    text = build_text(row)

    d_probs = get_probs(deberta_model, deberta_tokenizer, text)
    r_probs = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs = 0.7 * d_probs + 0.3 * r_probs

    top3_idx = np.argsort(w_probs)[::-1][:3]
    top3_letters = [label_letters[j] for j in top3_idx]

    scores.append(map_at_3(row["answer"], top3_letters))

final_map3 = np.mean(scores)
print("Q10 Answer:", round(final_map3, 4))

Q10 Answer: 1.0


In [134]:
import pandas as pd
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
print(sample_sub.columns.tolist())
sample_sub.head()

['ID', 'Prediction']


,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C


In [135]:
trainer.save_model("./deberta_finetuned")
tokenizer.save_pretrained("./deberta_finetuned")

('./deberta_finetuned/tokenizer_config.json',
 './deberta_finetuned/tokenizer.json')